In [1]:
import pandas as pd

df = pd.read_json("../data/EMMediaTopic-1.0.jsonl", lines=True)
print(df.shape)
print(df.columns.tolist())
df.head()

(21000, 5)
['document_id', 'lang', 'text', 'GPT-IPTC-label', 'split']


,document_id,lang,text,GPT-IPTC-label,split
0,CLASSLA-web.hr.4163187,hr,Osnovna škola Pećine Rijeka\n\nObavijest učeni...,education,dev
1,CLASSLA-web.hr.2821678,hr,U organizaciji Lige protiv raka Koprivničko-kr...,health,train
2,CLASSLA-web.hr.2508108,hr,[VIDEO] Rimac: Ova pobjeda u Beogradu znači pu...,sport,train
3,CLASSLA-web.hr.4605247,hr,Učenici prvih razreda naših škola iz Šarengrad...,education,train
4,CLASSLA-web.hr.2602058,hr,Poziv članovima Sindikata hrvatskih željezniča...,labour,train


In [2]:
df_el = df[df["lang"] == "el"].copy()
print("Ελληνικά άρθρα:", len(df_el))
print()
print(df["lang"].value_counts())
print()
print(df_el["split"].value_counts())
print()
print(df_el["GPT-IPTC-label"].value_counts())

Ελληνικά άρθρα: 5250

lang
hr    5250
sl    5250
ca    5250
el    5250
Name: count, dtype: int64

split
train    5000
dev       250
Name: count, dtype: int64

GPT-IPTC-label
sport                                        896
politics                                     787
economy, business and finance                663
arts, culture, entertainment and media       531
crime, law and justice                       420
human interest                               412
health                                       260
disaster, accident and emergency incident    245
education                                    211
conflict, war and peace                      172
labour                                       138
environment                                  117
science and technology                       117
society                                      111
religion                                      85
lifestyle and leisure                         53
weather                                   

In [3]:
df_el["n_words"] = df_el["text"].str.split().str.len()
print(df_el["n_words"].describe())
print()
print(df_el.iloc[0]["text"][:600])
print("→", df_el.iloc[0]["GPT-IPTC-label"])

count    5250.000000
mean      239.242667
std       143.951766
min        75.000000
25%       120.000000
50%       190.000000
75%       334.000000
max       512.000000
Name: n_words, dtype: float64

Ο Δήμος Ναυπακτίας για την επιστολή της Ομοσπονδίας Επαγγελματικών Βιοτεχνικών &amp; Εμπορικών Σωματείων Ναυπακτίας Δωρίδας στο δημοτικό συμβούλιο 

Δυτικά FM 

Η επιστολή της Ομοσπονδίας Επαγγελματικών Βιοτεχνικών &amp; Εμπορικών Σωματείων Ναυπακτίας και Δωρίδας με ημερομηνία 16-05-2017, σχετικά με το κυκλοφοριακό πρόβλημα της πόλης, πρωτοκολλήθηκε στο Δήμο την ίδια ημέρα. Η ημερήσια διάταξη του Δημοτικού Συμβουλίου της 17ης-5-2017, εκδόθηκε στις 12-05-2017 και ως εκ τούτου δεν ήταν δυνατόν να περιλαμβάνεται σ ́αυτήν το συγκεκριμένο θέμα. Ο Πρόεδρος της... 


→ politics


In [4]:
df_el.to_json(
    "../data/emmediatopic_el.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

In [5]:
import html

before = df_el["text"].str.contains(r"&amp;|&quot;|&#\d+;|&lt;|&gt;").sum()

df_el["text_clean"] = (
    df_el["text"]
    .map(html.unescape)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

after = df_el["text_clean"].str.contains(r"&amp;|&quot;|&#\d+;|&lt;|&gt;").sum()
print("Άρθρα με HTML entities - πριν:", before, "| μετά:", after)

Άρθρα με HTML entities - πριν: 351 | μετά: 0


In [6]:
from sklearn.model_selection import train_test_split

label_col = "GPT-IPTC-label"

train_df, temp_df = train_test_split(
    df_el, test_size=0.30, stratify=df_el[label_col], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[label_col], random_state=42
)

print(len(train_df), len(val_df), len(test_df))
print()
print(test_df[label_col].value_counts().tail(5))

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_json(f"../data/el_{name}.jsonl", orient="records", lines=True, force_ascii=False)

3675 787 788

GPT-IPTC-label
environment              17
society                  16
religion                 13
lifestyle and leisure     8
weather                   5
Name: count, dtype: int64
